# Analysis 19 — who writes like whom?

Character-trigram counts on equal windows of each author's text (log: `analysis-log.md`, Analysis 19).

Choices so far (student): messages glued per author with a line break, cut into 1,500-character
windows, the same number of windows per author, links replaced by `<URL>`, capitals kept.
Stage 3 option C: raw counts first, then read back which trigrams drive the distances before
deciding whether to drop or standardise anything.

In [ ]:
import re

import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Pipeline, TransformBase
from sklearn.feature_extraction.text import CountVectorizer

from wa_analyzer.data import load_own_chat

SEED = 42
WINDOW = 1_500  # characters per window (student)
SEPARATOR = "\n"  # between messages; Stage 6 reruns with " "
GLUED_SYSTEM_LINE = re.compile(r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - ")  # parser glitch; carries real names
NOT_A_TEXT = re.compile(r"<Media weggelaten>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand")
URL = re.compile(r"https?://\S+|www\.\S+")

## Clean

The lesson's `NormaliseText` lowercases and collapses all whitespace, which would undo two of the
choices above (capitals, line breaks). This step keeps both and only swaps links for a placeholder.

In [ ]:
class CleanForStyle(TransformBase):
    """Drop rows that are not typed text, and replace links with one placeholder."""

    def transform(self, data: pd.DataFrame, column: str, feature: str = "text") -> pd.DataFrame:
        text = data[column].fillna("").astype(str)
        typed = ~text.str.contains(GLUED_SYSTEM_LINE) & ~text.str.contains(NOT_A_TEXT) & (text.str.strip() != "")
        data = data[typed].copy()
        data[feature] = text[typed].str.replace(URL, "<URL>", regex=True)
        return data


own = load_own_chat().sort_values(["author", "timestamp"])
clean = Pipeline().add(CleanForStyle, column="message").apply(own)
print(f"{len(clean):,} of {len(own):,} messages kept")

## Windows

Each author's messages, in time order, glued into one text and cut at exactly `WINDOW` characters
(a window may start or end mid-message). The leftover at the end is dropped. Every author then
gets the same number of windows, drawn at random from all of theirs.

In [ ]:
def windows_per_author(frame: pd.DataFrame, size: int, separator: str) -> pd.DataFrame:
    """Cut each author's glued text into consecutive windows of exactly `size` characters."""
    rows = []
    for author, messages in frame.groupby("author"):
        glued = separator.join(messages.text)
        starts = np.cumsum([0] + [len(t) + len(separator) for t in messages.text[:-1]])  # where each message begins
        times = pd.to_datetime(messages.timestamp).to_numpy()
        for i in range(0, len(glued) - size + 1, size):
            first, last = np.searchsorted(starts, [i, i + size - 1], side="right") - 1
            rows.append({"author": author, "text": glued[i : i + size],
                         "middle": times[first] + (times[last] - times[first]) / 2})
    return pd.DataFrame(rows)


every_window = windows_per_author(clean, WINDOW, SEPARATOR)
per_author = every_window.author.value_counts()
n_each = per_author.min()
windows = every_window.groupby("author").sample(n_each, random_state=SEED).reset_index(drop=True)
print(f"windows per author before balancing:\n{per_author.to_string()}")
print(f"\nbalanced: {n_each} windows x {windows.author.nunique()} authors = {len(windows)}")

## Counts

`lowercase=False`: `CountVectorizer` lowercases by default, which would throw the capitals away.

In [ ]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)
counts = vectorizer.fit_transform(windows.text).toarray()
trigrams = vectorizer.get_feature_names_out()
print(f"{counts.shape[0]} windows x {counts.shape[1]:,} trigrams; a window uses {np.mean(counts > 0):.1%} of them")

## Which trigrams drive the distance? (option C)

The Manhattan distance between two windows is a sum over trigrams of |count difference|. Adding that
up over every pair of windows gives each trigram's share of all the distance in the data.
Sorting a column makes the pairwise sum cheap: the k-th smallest of n values appears with a plus sign
k−1 times and a minus sign n−k times.

In [ ]:
def pairwise_abs_sum(column_counts: np.ndarray) -> np.ndarray:
    """Per column: the sum of |x_i - x_j| over all pairs of rows."""
    n = column_counts.shape[0]
    weights = 2 * np.arange(1, n + 1) - n - 1
    return weights @ np.sort(column_counts, axis=0)


contribution = pairwise_abs_sum(counts)
readback = pd.DataFrame({
    "trigram": [repr(t) for t in trigrams],
    "share_of_distance": contribution / contribution.sum(),
    "mean_count": counts.mean(axis=0),
    "windows_with_it": (counts > 0).mean(axis=0),
}).sort_values("share_of_distance", ascending=False).reset_index(drop=True)

cumulative = readback.share_of_distance.cumsum()
for top in (10, 50, 200):
    print(f"top {top:>3} trigrams: {cumulative.iloc[top - 1]:.0%} of all distance")
print(f"trigrams needed for half of all distance: {int((cumulative < 0.5).sum()) + 1:,} of {len(readback):,}")
readback.head(25)

In [ ]:
rare = readback[readback.windows_with_it < 3 / len(windows)]
print(f"trigrams in fewer than 3 windows: {len(rare):,} ({len(rare) / len(readback):.0%} of all), "
      f"together {rare.share_of_distance.sum():.0%} of all distance")

## Step 1 — do the authors separate at all?

Silhouette on the Manhattan distances in the full trigram space, with the real author labels and
with shuffled ones (as `separation` in 06.2). If real is no better than shuffled, there is no style to group.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances

distance = manhattan_distances(counts, counts)
authors = windows.author.to_numpy()
rng = np.random.default_rng(SEED)
real = silhouette_score(distance, authors, metric="precomputed")
shuffled = [silhouette_score(distance, rng.permutation(authors), metric="precomputed") for _ in range(200)]
print(f"silhouette by author: {real:.3f}   shuffled labels: {np.mean(shuffled):.3f} "
      f"(highest of 200: {np.max(shuffled):.3f})")

# leave-one-out 1-NN: is each window's nearest other window by the same author? (chance = 1 in 9)
nearest = np.where(np.eye(len(distance), dtype=bool), np.inf, distance).argmin(axis=1)
print(f"1-NN, same author: {np.mean(authors[nearest] == authors):.0%}   (chance: {1 / len(set(authors)):.0%})")

## Stage 4 sketches

A: the obvious high-dimensional family — every window as a dot, coloured by author (PCA on the counts,
and t-SNE on the Manhattan distances). B: the alternative — one cell per pair of authors, the Manhattan
distance between their average windows (author centroids), so no projection in between.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2, random_state=SEED)
pca_xy = pca.fit_transform(counts)
tsne_xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=30,
               random_state=SEED).fit_transform(distance)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
palette = dict(zip(sorted(set(authors)), plt.get_cmap("tab10").colors))
for ax, xy, title in [
    (axes[0], pca_xy, f"PCA on counts ({pca.explained_variance_ratio_.sum():.0%} of variance in 2 components)"),
    (axes[1], tsne_xy, "t-SNE on Manhattan distances (perplexity 30)"),
]:
    for author, colour in palette.items():
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=14, color=colour, label=author, alpha=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
fig.suptitle("Sketch A — 306 windows of 1,500 characters, coloured by author")
fig.tight_layout()
fig.savefig("sketch-A-style-projection.png", dpi=120, bbox_inches="tight")

In [ ]:
centroids = pd.DataFrame(counts).groupby(authors).mean()
between = pd.DataFrame(manhattan_distances(centroids, centroids), index=centroids.index, columns=centroids.index)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
shown = between.where(~np.eye(len(between), dtype=bool))
image = ax.imshow(shown, cmap="viridis_r")
ax.set_xticks(range(len(between)), between.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(between)), between.index, fontsize=8)
for i in range(len(between)):
    for j in range(len(between)):
        if i != j:
            ax.text(j, i, f"{between.iat[i, j]:.0f}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(image, ax=ax, label="Manhattan distance between average windows (lower = more alike)")
ax.set_title("Sketch B — how far apart is each pair of authors?", fontsize=10)
fig.tight_layout()
fig.savefig("sketch-B-style-author-distances.png", dpi=120, bbox_inches="tight")

## Step 2 — are authors closer to their own background group?

Groups fixed in Stage 1, read from a gitignored file (pseudonym + study background could identify people).
Statistic: average centroid distance between two authors in the *same* group minus the average between
authors in *different* groups (negative = groups hang together). Chance: the same statistic for every one
of the 9!/(3!·4!·2!) = 1,260 ways to split 9 authors into groups of 3, 4 and 2.

In [ ]:
from itertools import combinations

groups = pd.read_csv("../../data/processed/author-groups.csv").set_index("author").group
assert set(groups.index) == set(between.index), "every author needs a group"  # noqa: S101


def within_minus_between(distances: pd.DataFrame, labels: pd.Series) -> float:
    """Mean distance of same-group author pairs minus mean distance of different-group pairs."""
    same, different = [], []
    for a, b in combinations(distances.index, 2):
        (same if labels[a] == labels[b] else different).append(distances.at[a, b])
    return float(np.mean(same) - np.mean(different))


def every_split(names: list[str], sizes: list[int]):
    """Yield every way to put `names` into groups of the given sizes, as a Series name -> group number."""
    if not sizes:
        yield pd.Series(dtype=int)
        return
    for chosen in combinations(names, sizes[0]):
        rest = [n for n in names if n not in chosen]
        for tail in every_split(rest, sizes[1:]):
            yield pd.concat([pd.Series(len(sizes), index=list(chosen)), tail])


observed = within_minus_between(between, groups)
splits = [within_minus_between(between, split) for split in every_split(list(between.index), [3, 4, 2])]
as_low = np.mean(np.array(splits) <= observed)
print(f"{len(splits):,} splits; real split: {observed:+.1f} (same-group pairs minus different-group pairs)")
print(f"splits that hang together at least as well as the real one: {as_low:.0%}")
print(f"range over all splits: {min(splits):+.1f} to {max(splits):+.1f}")

for name, members in groups.groupby(groups).groups.items():
    pairs = [between.at[a, b] for a, b in combinations(members, 2)]
    print(f"  {name:<10} ({len(members)} people): mean distance within {np.mean(pairs):.0f}")

## Perplexity sweep (6.6)

Colour = background group, marker = author. What should survive a sweep is which windows sit together;
positions, sizes and gaps will move.

In [ ]:
MARKERS = ["o", "s", "^", "D"]
GROUP_COLOURS = dict(zip(sorted(groups.unique()), plt.get_cmap("Set1").colors))
style = {}
for name, members in groups.groupby(groups).groups.items():
    for marker, author in zip(MARKERS, sorted(members)):
        style[author] = (GROUP_COLOURS[name], marker)

PERPLEXITIES = [5, 15, 30, 50]
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, perplexity in zip(axes.flat, PERPLEXITIES):
    xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=perplexity,
              random_state=SEED).fit_transform(distance)
    for author, (colour, marker) in style.items():
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=18, color=colour, marker=marker, alpha=0.8,
                   label=f"{groups[author]}: {author}")
    ax.set_title(f"perplexity {perplexity}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
handles, labels_ = axes.flat[0].get_legend_handles_labels()
order = np.argsort(labels_)
fig.legend([handles[i] for i in order], [labels_[i] for i in order], loc="lower center", ncol=3,
           fontsize=8, frameon=False)
fig.suptitle("Sketch C — t-SNE perplexity sweep, coloured by background group")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig("sketch-C-style-perplexity-sweep.png", dpi=110, bbox_inches="tight")

## Draft

Perplexity 15 (student): the per-person clumps are clear there. Colour = background group (validated palette,
colour-blind safe), shape = author. The numbers in the subtitle come from the full trigram space, not the picture.

In [ ]:
DRAFT_PERPLEXITY = 15
DRAFT_COLOURS = {"economics": "#2a78d6", "technical": "#eb6834", "education": "#1baf7a"}
same_author_share = np.mean(authors[nearest] == authors)

xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=DRAFT_PERPLEXITY,
          random_state=SEED).fit_transform(distance)

fig, ax = plt.subplots(figsize=(10, 8))
for group_name in DRAFT_COLOURS:
    members = sorted(groups[groups == group_name].index)
    for marker, author in zip(MARKERS, members):
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=34, marker=marker, color=DRAFT_COLOURS[group_name],
                   edgecolor="white", linewidth=0.6, label=f"{group_name}: {author}")
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_color("#c9c8c2")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9,
          title="study background: person", title_fontsize=9, alignment="left")
fig.suptitle("Group members have their own writing style,\nbut study background does not predict that style",
             x=0.02, ha="left", fontsize=14, fontweight="bold")
ax.set_title(
    f"Each dot is 1,500 characters of one person's messages. A dot's nearest neighbour is by the same person "
    f"{same_author_share:.0%} of the time (chance {1 / len(set(authors)):.0%}),\n"
    f"but people in the same study group are no closer to each other than in a random split "
    f"({as_low:.0%} of all 1,260 splits do as well).",
    loc="left", fontsize=9, color="#555550",
)
fig.text(0.02, 0.01,
         f"9 people, 34 windows each, counted as character trigrams. t-SNE (perplexity {DRAFT_PERPLEXITY}) keeps who sits "
         "together, not distances: clump sizes and gaps mean nothing.\nPer-person clumps also appear at perplexity "
         "5, 30 and 50. Numbers computed on the full counts, not on this picture.",
         fontsize=8, color="#555550", ha="left")
fig.tight_layout(rect=(0, 0.05, 1, 0.97))
fig.savefig("who-writes-like-whom-draft.png", dpi=150, bbox_inches="tight")

## Stage 6 — check A1: style or conversation?

Prediction written down first (student): 61% drops to about 50%; below 30% the title changes.
Each window may only match windows whose middle date is more than half a year away, so neighbouring
windows from the same conversation can't be the match. The rule is the same for every author.

In [ ]:
MIN_GAP = pd.Timedelta(days=182)

middles = windows.middle.to_numpy()
too_close = np.abs(middles[:, None] - middles[None, :]) <= MIN_GAP.to_timedelta64()
far_nearest = np.where(too_close, np.inf, distance).argmin(axis=1)
candidates = (~too_close).sum(axis=1)
own_candidates = np.array([np.sum(~too_close[i] & (authors == authors[i])) for i in range(len(authors))])
far_chance = np.mean(own_candidates / candidates)
print(f"1-NN, same author, all windows:          {same_author_share:.0%}")
print(f"1-NN, same author, > half a year apart:  {np.mean(authors[far_nearest] == authors):.0%}   "
      f"(chance with this rule: {far_chance:.0%})")
print(f"windows with no own-author window that far away: {np.sum(own_candidates == 0)}")

## Stage 6 — second grouping: lives in the city or not

Thought of after the draft, so it is comparison 2 of 2 and reported as such. Fixed before running (student):
city people write more alike (shorter messages, more laughing, more emoji); a finding only if fewer than 5%
of the 9!/(5!·4!) = 126 splits hang together at least as well.

In [ ]:
lives = pd.read_csv("../../data/processed/author-city.csv").set_index("author").lives
assert set(lives.index) == set(between.index), "every author needs a value"  # noqa: S101

city_observed = within_minus_between(between, lives)
city_splits = [within_minus_between(between, split) for split in every_split(list(between.index), [5, 4])]
city_as_low = np.mean(np.array(city_splits) <= city_observed)
print(f"{len(city_splits)} splits; real split: {city_observed:+.1f}")
print(f"splits that hang together at least as well as the real one: {city_as_low:.0%}")
for name, members in lives.groupby(lives).groups.items():
    pairs = [between.at[a, b] for a, b in combinations(members, 2)]
    print(f"  {name:<9} ({len(members)} people): mean distance within {np.mean(pairs):.0f}")